# SF/TF grating onset profile

Onset-aligned dF/F of the SFTF grating presentations, used to choose how many volumes to skip at the
start of each trial (`gratings.generate_trials_df(skip_first_n_volumes=...)` and `LAG` in
`figures/fig_reviewer_gratings.ipynb`).

- Gratings last 4 s and follow each other with no grey screen in between; imaging is a single plane at 15.2 Hz
  (1 volume = 66 ms, ~61 volumes per trial). Indicator: GCaMP6f.
- Selected ROIs: one-way ANOVA across the grating conditions, p < 0.01, on the full-trial mean, and mean
  dF/F over the 4 s trial at the preferred condition > 0.3 (even repeats, i.e. the averaged trials).
- Each ROI's preferred condition is chosen on odd repeats and its trace averaged over even repeats, so the
  selection does not inflate the trace.

**Result (Oct 2026):** no onset transient. The response rises to a plateau (50% at 0.33 s, plateau at about 1 s)
and decays after offset (to 50% at 0.34 s, to 10% at 0.9 s). The start of each trial therefore holds the rise
of the new response and the decay of the previous one. Skip chosen: **8 volumes (0.53 s)**, the volume
closest to 0.5 s: the response is at 75% of its plateau and the previous one has decayed to 22%; the window
ends at the next onset. Used by all trial-averaged SF/TF analyses (`gratings.generate_trials_df`,
`fig_reviewer_gratings`, `precompute_data/sftf_trial_average_subsets.py`).

**Bug fixed at the same time:** the earlier window was `[onset + 2, next_onset + 2)`. Both edges were shifted,
so each trial skipped only 0.13 s and included the first 0.13 s of the next grating (figure 2).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import flexiznam as flz

from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from cottage_analysis.preprocessing import synchronisation
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
PHOTODIODE_PROTOCOL = 5
TRACES = {"anatomical_only": 3, "ast_neuropil": False}
PARAMS = ["SpatialFrequency", "TemporalFrequency", "Angle"]
FS = 15.2  # volumes per second (single plane)
TRIAL_S = 4.0  # grating duration
PRE, POST = 30, 120  # volumes before / after onset in the snippets (-2 s to +8 s)
P_RESPONSIVE = 0.01
MIN_PREF_DFF = 0.3  # mean dF/F over the 4 s trial at the preferred condition
OLD_SKIP = 2  # before the fix: window [onset + 2, next_onset + 2)
NEW_SKIP = 8  # after the fix: window [onset + 8, next_onset)
REBUILD = False

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="sftf_onset_profile")
CACHE = SAVE_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

## Onset-aligned traces

In [ ]:
def recording_snippets(recording):
    # dF/F snippets around each grating onset (trial, time, roi) and full-trial means (trial, roi).
    vs_df = synchronisation.generate_vs_df(
        recording=recording,
        photodiode_protocol=PHOTODIODE_PROTOCOL,
        flexilims_session=flm,
        project=PROJECT,
    )
    img_df = synchronisation.generate_imaging_df(
        vs_df=vs_df,
        recording=recording,
        flexilims_session=flm,
        filter_datasets=TRACES,
        return_volumes=True,
    ).reset_index(drop=True)
    dff = np.vstack(img_df.dffs.values)
    stim = img_df["stimulus_harptime"]
    onset = (stim.notna() & (stim != stim.shift())).values
    starts = np.flatnonzero(onset)
    trials = img_df.loc[onset, PARAMS].reset_index(drop=True)
    ok = (starts - PRE >= 0) & (starts + POST <= len(dff))
    snips = np.stack([dff[s - PRE : s + POST] for s in starts[ok]])
    ends = np.append(starts[1:], starts[-1] + int(np.median(np.diff(starts))))
    resp = np.stack([dff[s:e].mean(axis=0) for s, e in zip(starts, ends)])
    return trials[ok].reset_index(drop=True), snips.astype(np.float32), resp[ok]


def session_profile(session):
    # Per-ROI trace at the preferred condition (even repeats), mean over all gratings, ANOVA p.
    path = CACHE / f"{session}_onset_profile.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    recs = flz.get_children(
        parent_name=session, children_datatype="recording", flexilims_session=flm
    )
    recs = recs[recs["name"].str.contains("SFTF")].sort_index()
    out = [recording_snippets(rec) for _, rec in recs.iterrows()]
    trials = pd.concat([o[0] for o in out], ignore_index=True)
    snips = np.concatenate([o[1] for o in out])
    resp = np.concatenate([o[2] for o in out])

    cond = trials.groupby(PARAMS).ngroup().values
    rep = trials.groupby(PARAMS).cumcount().values
    odd, even = rep % 2 == 1, rep % 2 == 0
    n_cond, n_roi = cond.max() + 1, resp.shape[1]
    mean_odd = np.stack([resp[odd & (cond == c)].mean(0) for c in range(n_cond)])
    pref = mean_odd.argmax(0)
    pval = stats.f_oneway(*[resp[cond == c] for c in range(n_cond)]).pvalue
    pref_trace = np.stack(
        [snips[even & (cond == pref[r]), :, r].mean(0) for r in range(n_roi)]
    )
    data = dict(pref_trace=pref_trace, all_trace=snips.mean(0).T, anova_p=pval)
    pd.to_pickle(data, path)
    return data


profiles = {s: session_profile(s) for s in SESSIONS}
t = (np.arange(PRE + POST) - PRE) / FS
in_trial = (t >= 0) & (t < TRIAL_S)


def selected(d):
    # responsive and strongly driven at the preferred condition
    return (d["anova_p"] < P_RESPONSIVE) & (d["pref_trace"][:, in_trial].mean(1) > MIN_PREF_DFF)


n_resp = sum((d["anova_p"] < P_RESPONSIVE).sum() for d in profiles.values())
pref = np.concatenate([d["pref_trace"][selected(d)] for d in profiles.values()])
allt = np.concatenate([d["all_trace"][selected(d)] for d in profiles.values()])
print(f"{len(pref)} selected ROIs out of {n_resp} responsive")

## Rise and decay times

Per ROI: trace minus its value at onset, normalised to its peak within the trial. Population: mean trace with
the pre-onset level as 0 and the 2–4 s plateau as 1.

In [ ]:
rel = pref - pref[:, [PRE]]
peak_i = np.argmax(np.where(in_trial, rel, -np.inf), axis=1)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = rel / rel[np.arange(len(rel)), peak_i][:, None]
t_peak = t[peak_i]
t_half = np.array([t[PRE + np.argmax(n[PRE:] >= 0.5)] for n in norm])

pop = pref.mean(0)
base = pop[t < 0].mean()
plateau = pop[(t >= 2) & (t < TRIAL_S)].mean()
pop_norm = (pop - base) / (plateau - base)
after = t >= 0
off = t >= TRIAL_S
timing = pd.Series(
    {
        "rise to 10% (s)": t[after][np.argmax(pop_norm[after] >= 0.1)],
        "rise to 50% (s)": t[after][np.argmax(pop_norm[after] >= 0.5)],
        "rise to 90% (s)": t[after][np.argmax(pop_norm[after] >= 0.9)],
        "decay to 50% after offset (s)": t[off][np.argmax(pop_norm[off] <= 0.5)] - TRIAL_S,
        "decay to 10% after offset (s)": t[off][np.argmax(pop_norm[off] <= 0.1)] - TRIAL_S,
        "median ROI time to 50% (s)": np.median(t_half),
    }
).round(2)
timing

## Figure 1: onset profile

- **A**: mean ± SEM dF/F at the preferred condition (black) and over all gratings (grey). Dashed lines: grating
  onset and offset (the next grating starts at 4 s). Orange: the 2 volumes skipped before the fix.
- **B**: per-ROI normalised trace, median and IQR.
- **C**: time for each ROI to reach 50% of its peak.
- **D**: time to peak. Not informative: responses plateau, so the peak falls wherever the noise is highest.

In [ ]:
def mark_trial(ax):
    for x in (0, TRIAL_S):
        ax.axvline(x, color="#8a8984", lw=0.5, ls="--")
    ax.set_xlabel("Time from grating onset (s)")
    ax.set_xlim(t[0], t[-1])


fig, axes = plt.subplots(
    1, 4, figsize=(20 * CM, 5 * CM), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1, 1])
)
ax = axes[0]
for tr, color, label in [(pref, "k", "preferred (even reps)"), (allt, "#8a8984", "all gratings")]:
    m, e = tr.mean(0), tr.std(0) / np.sqrt(len(tr))
    ax.fill_between(t, m - e, m + e, color=color, alpha=0.25, lw=0)
    ax.plot(t, m, color=color, lw=1, label=label)
ax.set_ylabel("dF/F")
ax.legend(frameon=False, fontsize=style.FONTSIZE_DICT["legend"], loc="center right")
ax = axes[1]
ax.fill_between(t, *np.nanpercentile(norm, [25, 75], axis=0), color="k", alpha=0.2, lw=0)
ax.plot(t, np.nanmedian(norm, 0), "k", lw=1)
ax.axhline(0, color="#8a8984", lw=0.5)
ax.set_ylabel("Normalised dF/F\n(median, IQR)")
for ax in axes[:2]:
    mark_trial(ax)
    ax.axvspan(0, OLD_SKIP / FS, color="#eb6834", alpha=0.3, lw=0)
bins = np.arange(0, TRIAL_S + 1 / FS, 1 / FS) - 0.5 / FS
for ax, v, label in [
    (axes[2], t_half, "Time to 50% of peak (s)"),
    (axes[3], t_peak, "Time to peak (s)"),
]:
    ax.hist(v, bins=bins, color="k")
    ax.axvline(np.median(v), color="#eb6834", lw=1)
    ax.set_title(f"median {np.median(v):.2f} s", fontsize=style.FONTSIZE_DICT["title"])
    ax.set_xlabel(label)
    ax.set_xlim(-0.1, TRIAL_S)
axes[2].set_ylabel("Number of ROIs")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(
    f"{len(pref)} ROIs (ANOVA p < {P_RESPONSIVE}, preferred dF/F > {MIN_PREF_DFF}), {len(SESSIONS)} sessions",
    fontsize=style.FONTSIZE_DICT["title"],
)
fig.tight_layout()
style.savefig(SAVE_ROOT / "sftf_onset_profile.pdf", fig=fig, bbox_inches="tight")

## Figure 2: response window before and after the fix

Population mean (normalised as above) with the averaging window of one trial:

- **before** (orange): `[onset + 2, next_onset + 2)`, 0.13–4.13 s. Starts during the rise and the previous
  grating's decay, and ends 0.13 s into the next grating.
- **after** (blue): `[onset + 8, next_onset)`, 0.53–4.0 s.

The next grating's onset (4 s) is a random condition, so in the population average the trace after 4 s is the
decay of this trial's response, and the part of the next trial's own response is not visible. For a single
trial, the old window's last 0.13 s are the next grating's rise and the end of this grating's decay mixed.

In [ ]:
windows = {
    "before fix": (OLD_SKIP / FS, TRIAL_S + OLD_SKIP / FS, "#eb6834"),
    "after fix": (NEW_SKIP / FS, TRIAL_S, "#2a78d6"),
}
fig, ax = plt.subplots(figsize=(10 * CM, 5.5 * CM))
for i, (label, (t0, t1, color)) in enumerate(windows.items()):
    ax.axvspan(t0, t1, color=color, alpha=0.12, lw=0)
    y = 1.2 + 0.22 * i
    ax.plot([t0, t1], [y, y], color=color, lw=3, solid_capstyle="butt")
    ax.text(t0, y + 0.04, f"{label}: {t0:.2f}–{t1:.2f} s", va="bottom", color=color,
            fontsize=style.FONTSIZE_DICT["legend"])
ax.plot(t, pop_norm, "k", lw=1)
ax.axhline(0, color="#8a8984", lw=0.5)
ax.axhline(1, color="#8a8984", lw=0.5, ls=":")
mark_trial(ax)
ax.set_xlim(-1, 6)
ax.set_ylabel("dF/F (0: pre-onset, 1: plateau)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "sftf_window_before_after_fix.pdf", fig=fig, bbox_inches="tight")

# mean normalised response inside each window
pd.Series(
    {label: pop_norm[(t >= t0) & (t < t1)].mean() for label, (t0, t1, _) in windows.items()},
    name="mean normalised dF/F in window",
).round(3)